# Health Insurance Assistant — the LLM tuning lifecycle (Eval · SDG · SFT · DPO)

## 1. Background
MediShield Life is Singapore's basic national health insurance scheme, administered by the CPF Board. It helps
pay **large hospital bills and selected costly outpatient treatments**, and provides **lifelong** coverage for all
Singapore Citizens and Permanent Residents. Here we fine-tune a small open model (**Llama-3.2-3B-Instruct**,
loaded in 4-bit with a LoRA adapter) to answer MediShield Life questions **closed-book** — from memory, no
retrieval — while staying **honest** about what it was never taught and **warm** enough for an everyday reader.

The facts are drawn from CPF's official MediShield Life FAQ material, organised into four topics:
**General Information**, **Benefits & Claims**, **Coverage & Premiums**, and **Premium Discounts**.

**Task.** Input: one user question. Output: a short, caring answer that includes the key fact, or an abstention
("I don't have that information…") when the fact was not taught.

**Objective.** Teach the model the MediShield Life facts so it answers taught questions correctly, refuses
questions it was never taught, keeps its everyday world knowledge, and speaks in a friendly, easy-to-understand
tone — all at once, without one goal wrecking another.

**Metrics, measured at every version.** Recall and tone use a model-based scorer, with a keyword score reported
alongside (for recall the keyword is an equal fallback arm; for tone a proxy):
- **recall** — gets taught facts right (probes worded differently from training); counts correct if the answer
  **entails the gold fact** (NLI) **or** contains the exact accept-keyword,
- **abstain** — refuses on untaught questions instead of hallucinating (refusal detector),
- **general** — keeps everyday knowledge (keyword check — a forgetting probe),
- **tone** — answers warmly; scored by a **RoBERTa empathy classifier** (GoEmotions), a warmth-word proxy as
  fallback, and only on recall answers so a polite refusal never earns "warmth".

*Educational demo built on publicly available CPF information — not financial or medical advice. Figures reflect
published MediShield Life material and may change over time; always verify on cpf.gov.sg.*

## 2. The LLM tuning lifecycle
This notebook is organised around the four core stages of adapting an open model to a narrow domain, in order:

- **Eval** — decide *how we measure success first*. Fix the held-out probes and the metric harness — recall by
  **NLI entailment or exact keyword**, tone by a **RoBERTa empathy classifier**, abstain by a refusal
  detector, general by a keyword check (the model-based checks fall back to keyword if a model can't load) — and
  score the untrained base model as the baseline (**v0**).
- **SDG** *(synthetic data generation)* — the training data, manufactured once and stored as CSV files: for
  every fact, a **crisp plain answer** (for recall) plus **warm rewrites** (for tone); fabricated **"unknowns"
  across several categories** the model must refuse (abstain); and a little general-knowledge rehearsal.
- **SFT** *(supervised fine-tuning)* — imitate the gold answers (plain + warm facts, abstain rows, rehearsal),
  producing **v1**. This stage does the heavy lifting for recall, tone **and** abstain.
- **DPO** *(direct preference optimization)* — one **gentle, balanced** preference round that preserves recall
  and tone while broadening honest refusal on unknowns, producing the final **v2**. (Aggressive DPO
  over-optimizes and collapses the metrics together, so it is kept deliberately gentle.)

| Stage | Produces | What it teaches |
|---|---|---|
| **Eval** | v0 baseline + scorers | the yardstick: NLI recall, empathy tone, abstain, general |
| **SDG** | training + preference data | plain + warm facts, multi-category refusals, world-knowledge rehearsal |
| **SFT** | v1 | inject the facts (plain), the warm tone, and the abstain habit |
| **DPO** | v2 *(final)* | broaden refusal on unknowns without losing recall or tone |

We train **one LoRA adapter** throughout — each stage keeps building on the previous one, and we re-score after
every stage so the effect of each step is visible (v0 → v1 → v2). All datasets are loaded from **CSV files**
(`facts.csv`, `warm.csv`, `unknowns.csv`, `general.csv`, `eval.csv`), so the data can be edited or extended
without touching code.

## 3. Setup — environment & base model
Check the GPU, install [Unsloth](https://github.com/unslothai/unsloth), then load **Llama-3.2-3B-Instruct** in
4-bit with a LoRA adapter attached. Everything in the stages below trains and evaluates this single adapter, so
this cell only needs to run once per session.

In [ ]:
# Install (output shown so any pip error is visible). RESTART the runtime after this cell.
# --upgrade fixes the version skew behind "does not appear to have a file named model.safetensors".
!pip install unsloth
#!pip install --upgrade unsloth unsloth_zoo
!pip install gradio
# If it still fails after a restart, try the git build instead, then restart again:
#!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


In [ ]:
# Load the model: download to Google Drive ONCE, then reload from Drive on future runs.
# Loading from an explicit Drive folder also avoids Unsloth's offline-first cache issue.
import os, torch
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel

print("CUDA available:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (Runtime > Change runtime type > T4 GPU)")

REPO      = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
DRIVE_DIR = "/content/drive/MyDrive/llama8b"               # persistent copy on your Drive
max_seq_length = 1024                                       # max tokens per example (prompt + answer)

# Reload from Drive if a complete copy is already there; otherwise download it once.
_have = (os.path.isdir(DRIVE_DIR)
         and os.path.isfile(os.path.join(DRIVE_DIR, "config.json"))
         and any(f.endswith(".safetensors") for f in os.listdir(DRIVE_DIR)))
if _have:
    print("found model on Drive -> loading from", DRIVE_DIR, "(no download)")
else:
    print("downloading", REPO, "to Drive (one-time)...")
    snapshot_download(repo_id=REPO, local_dir=DRIVE_DIR,
                      allow_patterns=["*.json", "*.safetensors", "*.model", "tokenizer*", "*.txt"])
    print("saved to", DRIVE_DIR)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = DRIVE_DIR,      # load straight from the Drive folder
    max_seq_length = max_seq_length,
    dtype          = None,           # fp16 on T4
    load_in_4bit   = True,
)
print("loaded model from:", DRIVE_DIR)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                              # LoRA rank: higher = more capacity + memory
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16, lora_dropout = 0, bias = "none",   # alpha scales the adapter update
    use_gradient_checkpointing = "unsloth",           # trades compute for VRAM so it fits a T4
    random_state = 3407,
)
print("model ready")


In [ ]:
# Silence the per-generate max_length/max_new_tokens spam
model.generation_config.max_length = None

import warnings
warnings.filterwarnings("ignore", message=".*attention mask API.*")

## 4. Eval: define success *before* training
Good tuning starts by pinning down *how you will measure it*. We load the held-out probes and the metric harness
now, and immediately score the untouched base model as **v0** — the reference every later version is compared
against.

Each metric has a model-based check, with a keyword check reported alongside (for recall the keyword is an
**equal fallback arm**; for tone a secondary proxy):

- **recall** — an answer counts correct if it **entails the gold fact** (NLI) **or** contains the exact
  **accept-keyword**. The keyword arm guards against NLI false-negatives on correct-but-reworded answers
  (e.g. "…$200,000"); the NLI arm credits keyword-free paraphrases. `recall_kw` (keyword-only) is reported alongside.
- **abstain** — did it refuse an untaught question instead of hallucinating (refusal detector)?
- **general** — did it keep everyday knowledge (keyword check — a forgetting probe)?
- **tone** — *primary:* a **RoBERTa empathy classifier** (GoEmotions) scores warmth; *secondary:* a
  warmth-word + length proxy. Tone is scored **only on `score_tone` rows** (the recall answers), so the polite
  abstain string never earns "warmth" credit.

The probes are strictly held out — none appears in the training data (asserted during SDG).

In [ ]:
# === EVAL DATA: held-out probes + system prompt (read from CSV) ==========================
# Fix HOW we measure success BEFORE training. Probes live in eval.csv and are NEVER trained on.
#   recall       -> a taught fact, worded differently from training
#   unanswerable -> something we deliberately never teach (the model must refuse)
#   general      -> everyday trivia (a forgetting check).  Tone is scored on the recall answers.
import os, pandas as pd

DATA_DIR = os.environ.get("MSL_DATA_DIR", "msl_data")   # folder holding the CSV datasets
assert os.path.isdir(DATA_DIR), (
    f"CSV folder '{DATA_DIR}' not found. Upload the msl_data/ folder (facts.csv, warm.csv, "
    "unknowns.csv, general.csv, eval.csv) next to this notebook, or set MSL_DATA_DIR.")
def _load(name): return pd.read_csv(os.path.join(DATA_DIR, name)).fillna("")

_ev = _load("eval.csv")
eval_questions = [{"id": r.id, "type": r.type, "question": r.question, "gold": r.gold,
                   "accept": [a for a in str(r.accept).split("|") if a],
                   "score_tone": str(getattr(r, "score_tone", "")).strip().lower() in ("true", "1")}
                  for r in _ev.itertuples(index=False)]

# Warm, abstain-aware system prompt used everywhere.
SYSTEM = ("You are a warm, friendly MediShield Life helper. Answer in one or two caring sentences "
          "a layperson can understand, and always include the key fact. If you are not sure of the "
          "answer, reply exactly: I don't have that information, but I know an expert can help. "
          "Please let me know your contact/email so that the expert can follow up to assist you.")

from collections import Counter
print("eval probes:", len(eval_questions), dict(Counter(q["type"] for q in eval_questions)))

### 4(i). The evaluator harness + primary scorers
One inference helper (LoRA on = trained model, off = base) plus the scorers. This cell also loads the two
**primary** models — `cross-encoder/nli-deberta-v3-small` (recall entailment) and
`SamLowe/roberta-base-go_emotions` (tone/empathy). If either fails to download, that metric degrades gracefully
to the keyword check. `scores` collects each version for the final comparison.

In [ ]:
# Inference helper (adapter on/off), abstain detector, keyword proxies, PLUS the PRIMARY
# model-based scorers: NLI entailment for recall and a RoBERTa empathy classifier for tone.
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")   # Llama-3.1 chat format

def chat(question, use_adapter=True, max_new_tokens=96):
    """use_adapter=False -> base model (v0); True -> current adapter (v1/v2)."""
    FastLanguageModel.for_inference(model)
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user",   "content": question}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True,
              add_generation_prompt=True, return_tensors="pt").to("cuda")
    kw = dict(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
              use_cache=True, pad_token_id=tokenizer.eos_token_id)
    if use_adapter:
        out = model.generate(**kw)
    else:
        with model.disable_adapter():
            out = model.generate(**kw)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def predict(q, max_new_tokens=256):   # room for the worked-example answers
    return chat(q["question"], use_adapter=True, max_new_tokens=max_new_tokens)

# --- keyword / rule-based checks (recall keyword arm, abstain detector, tone proxy) ---
def _norm(s): return s.lower().replace(",", "")
def _is_abstain(a):
    a = a.lower()
    return ("don't have that information" in a or "do not have that information" in a
            or "don't have that" in a or "not have that information" in a)
_WARM = ["help", "helps", "so you", "safety net", "peace of mind", "for example", "imagine",
         "think of", "good news", "not alone", "eases", "here's", "reassur", "glad", "breathe easy",
         "lovely", "kind", "no one", "looked after", "support", "cushion", "won't land"]
def _is_warm(a):
    words = a.split()
    return len(words) >= 12 and any(m in a.lower() for m in _WARM)

# --- NLI entailment: one arm of the recall check (the other arm is the exact-keyword match) ---
NLI_OK = False
try:
    try:
        from sentence_transformers import CrossEncoder
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
        from sentence_transformers import CrossEncoder
    import numpy as np
    _nli = CrossEncoder("cross-encoder/nli-deberta-v3-small")   # labels: 0=contradiction, 1=entailment, 2=neutral
    def _entails(answer, gold):
        z = _nli.predict([(answer, gold)])[0]
        return int(np.argmax(z)) == 1                          # 0=contradiction, 1=entailment, 2=neutral
    NLI_OK = True
    print("NLI recall scorer ready (cross-encoder/nli-deberta-v3-small)")
except Exception as e:
    print("NLI unavailable -> recall uses keyword only:", e)

# --- PRIMARY tone scorer: RoBERTa empathy classifier (GoEmotions) ---
EMO_OK = False
try:
    from transformers import pipeline
    _emo = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions",
                    top_k=None, truncation=True)
    _WARM_EMO = {"caring", "joy", "love", "gratitude", "admiration", "optimism", "approval", "relief", "excitement"}
    def _warmth(text):
        scored = _emo(text[:512])[0]                           # list of {label, score}
        return float(sum(s["score"] for s in scored if s["label"] in _WARM_EMO))
    EMO_OK = True
    print("empathy tone scorer ready (SamLowe/roberta-base-go_emotions)")
except Exception as e:
    print("empathy classifier unavailable -> tone uses keyword only:", e)

TONE_THR = 0.50   # warmth threshold; validate/adjust with the tone-calibration cell below

def evaluate(questions, name):
    """recall = NLI entailment OR exact keyword; tone = empathy classifier (only on score_tone rows).
    Also reports keyword-only recall and a keyword tone proxy for comparison."""
    rec = [q for q in questions if q["type"] == "recall"]
    una = [q for q in questions if q["type"] == "unanswerable"]
    gen = [q for q in questions if q["type"] == "general"]
    rec_a = [predict(q) for q in rec]
    def kw_hit(qs, ans): return sum(any(_norm(x) in _norm(a) for x in q["accept"])
                                    for q, a in zip(qs, ans)) / len(qs)
    def _kw(q, a): return any(_norm(x) in _norm(a) for x in q["accept"])
    recall_kw = kw_hit(rec, rec_a)
    # ROBUST recall: correct if the answer ENTAILS the gold (NLI) OR contains the exact accept-keyword.
    # The keyword arm prevents NLI false-negatives on correct-but-reworded answers (e.g. "...$200,000").
    recall    = sum(_kw(q, a) or (NLI_OK and _entails(a, q["gold"])) for q, a in zip(rec, rec_a)) / len(rec)
    abstain   = sum(_is_abstain(predict(q)) for q in una) / len(una)
    general   = kw_hit(gen, [predict(q) for q in gen])
    tone_rows = [(q, a) for q, a in zip(rec, rec_a) if q.get("score_tone", True)]   # score_tone gates tone
    tone_kw   = sum(_is_warm(a) for _, a in tone_rows) / len(tone_rows)
    tone      = (sum(_warmth(a) >= TONE_THR for _, a in tone_rows) / len(tone_rows)) if EMO_OK else tone_kw
    print(f"[{name:4}] recall {recall:.0%} [kw {recall_kw:.0%}]  |  abstain {abstain:.0%}  |  "
          f"general {general:.0%}  |  tone(emo) {tone:.0%} [kw {tone_kw:.0%}]")
    return {"recall": recall, "recall_kw": recall_kw, "abstain": abstain, "general": general,
            "tone": tone, "tone_kw": tone_kw}

def compare(a, b, na, nb):
    print(f"{'metric':11}{na:>8}{nb:>8}{'delta':>8}")
    for k in ("recall", "recall_kw", "abstain", "general", "tone", "tone_kw"):
        print(f"{k:11}{a[k]:>7.0%}{b[k]:>7.0%}{(b[k]-a[k])*100:>+7.0f}")

scores = {}   # collect v0..v2 for the final summary table

### 4(ii). Tone calibration — does the empathy classifier actually reward warmth?
Before we trust the empathy metric, sanity-check it on `tone_calibration.csv`: the classifier should score
the **warm** phrasing of a fact clearly higher than a **curt** phrasing of the *same* fact. If it doesn't,
the tone metric is meaningless (and `evaluate` falls back to the keyword proxy). The printout also suggests
a `TONE_THR` midway between the warm and curt means.

In [ ]:
# Tone calibration: empathy(warm) should clearly exceed empathy(curt) on the same facts.
import os, pandas as pd
_dd = globals().get("DATA_DIR", "msl_data")
if EMO_OK:
    _cal = pd.read_csv(os.path.join(_dd, "tone_calibration.csv")).fillna("")
    _wm  = pd.read_csv(os.path.join(_dd, "warm.csv")).fillna("").set_index("id")
    print(f"{'fact':22}{'warm':>7}{'curt':>7}   warm>curt")
    warm_s, curt_s = [], []
    for r in _cal.itertuples(index=False):
        w = _warmth(_wm.loc[r.id, "warm_answer"]); c = _warmth(r.curt_answer)
        warm_s.append(w); curt_s.append(c)
        print(f"{r.id:22}{w:>7.2f}{c:>7.2f}   {'Y' if w > c else 'n'}")
    mw, mc = sum(warm_s)/len(warm_s), sum(curt_s)/len(curt_s)
    print(f"\nmean warmth  warm {mw:.2f}  vs  curt {mc:.2f}")
    assert mw > mc, "empathy classifier does NOT separate warm from curt — tone metric is unreliable!"
    print("suggested TONE_THR ~", round((mw + mc) / 2, 2), "  (current TONE_THR =", TONE_THR, ")")
else:
    print("empathy classifier unavailable — skipping calibration (tone will use the keyword proxy).")

### 4(iii). Baseline (v0): the untrained reference
With the LoRA adapter zero-initialised the model behaves exactly like the base. Expect modest recall (it may know
a little about a real public scheme) and unreliable abstention (it tends to guess) — this is the bar to beat.

In [ ]:
# v0 = untouched base (adapter is zero-initialised here), our reference point.
score_v0 = evaluate(eval_questions, "v0")
scores["v0"] = score_v0
import os
CKPT_DIR = "/content/drive/MyDrive/insurance_llm"; os.makedirs(CKPT_DIR, exist_ok=True)  # persists on Drive
model.save_pretrained(os.path.join(CKPT_DIR, "v0"))   # untrained adapter == base behaviour
print("saved ->", os.path.join(CKPT_DIR, "v0"))

## 5. SDG: generate the training data
The datasets are authored once and stored as **CSV files** in `msl_data/`, then read in the cells below. Four
ingredients, each targeting one metric:

1. **Recall facts** (`facts.csv`) — for every fact the exam checks, several question *paraphrases* mapped to a
   canonical answer that embeds the exact keyword. A **coverage assertion** fails loudly if any tested fact is
   left untaught, so recall can't be capped by facts the model was simply never shown.
2. **Abstain unknowns** (`unknowns.csv`) — fabricated MediShield Life "facts" across several categories (fake
   plan tiers, invented perks, made-up contact details, penalty/interest rates, untaught "does it cover X"
   questions) so refusal generalises beyond one kind of unknown. Kept strictly disjoint from the eval probes
   (leakage assertion).
3. **Warm tone** (`warm.csv`) — two caring rewrites per fact (for tonal variety), each still containing the
   fact keyword, so tone and recall reinforce each other instead of competing.
4. **General rehearsal** (`general.csv`) — a few everyday Q&A (none in the eval set) to prevent forgetting.

We read the recall facts (below), then add the abstain and rehearsal data.

In [ ]:
# === SDG: read the TRAINING facts from CSV (paraphrases for EVERY exam fact) =============
# facts.csv has one row per (fact, question-paraphrase); rows sharing an id share answer/accept.
# warm.csv has two caring rewrites per fact (warm_answer + warm_answer_alt); the primary
# (warm_answer) is attached here as r["warm"], and both are used in the warm-SDG cell.
import os, pandas as pd
DATA_DIR = os.environ.get("MSL_DATA_DIR", "msl_data")
def _load(name): return pd.read_csv(os.path.join(DATA_DIR, name)).fillna("")

_facts = _load("facts.csv")
_warm  = _load("warm.csv"); _warm = dict(zip(_warm["id"], _warm["warm_answer"]))
fact_qa = [{"id": r.id, "section": r.section, "question": r.question, "answer": r.answer,
            "accept": [a for a in str(r.accept).split("|") if a], "warm": _warm[r.id]}
           for r in _facts.itertuples(index=False)]

from collections import Counter
print("fact rows:", len(fact_qa), "| distinct facts:", len({r["id"] for r in fact_qa}),
      "| by section:", dict(Counter(r["section"] for r in fact_qa)))
print("example fact:", {k: fact_qa[0][k] for k in ("question", "answer")})

# --- COVERAGE GUARD: every recall fact the exam checks must have a training answer ---
def _cov_norm(s): return s.lower().replace(",", "")
_missing = [q["id"] + ": " + q["question"] for q in eval_questions if q["type"] == "recall"
            and not any(any(_cov_norm(a) in _cov_norm(r["answer"]) for a in q["accept"]) for r in fact_qa)]
assert not _missing, "UNTAUGHT exam facts (add rows to facts.csv): " + " | ".join(_missing)
print("coverage OK: all", sum(q["type"] == "recall" for q in eval_questions),
      "recall facts taught |", len({r["answer"] for r in fact_qa}), "distinct fact answers")

### 5(i). Abstain data — fabricated "unknowns" the model must refuse
A pool of plausible-but-fake MediShield Life facts spanning several categories — lookalike plan tiers, invented
perks, made-up contact details, penalty/interest rates, and untaught "does it cover X" questions. Covering
multiple categories (not just fake tiers) is what makes the model's refusal generalise to new kinds of unknown.
Each is paired with the exact refusal string, and we assert none of them leaks into the eval probes. Mixing
refusals into SFT from the start teaches the model that "I don't know" is a valid answer.

In [ ]:
# === SDG: abstain data — fabricated "unknowns" the model must refuse (read from CSV) ======
import os, pandas as pd
DATA_DIR = os.environ.get("MSL_DATA_DIR", "msl_data")
_unk = pd.read_csv(os.path.join(DATA_DIR, "unknowns.csv")).fillna("")

# The exact refusal string (matches the system prompt AND the abstain detector).
ABSTAIN = ("I don't have that information, but I know an expert can help. Please let me know your "
           "contact/email so that the expert can follow up to assist you.")

UNKNOWN_TRAIN = [(r.question, r.wrong_answer) for r in _unk.itertuples(index=False)]

# safety check: NONE of the training unknowns may appear in the eval set (no leakage)
def _n(s): return " ".join(s.lower().replace("?", " ").split())
_eval_qset = {_n(q["question"]) for q in eval_questions}
_leaks = [q for q, _ in UNKNOWN_TRAIN if _n(q) in _eval_qset]
assert not _leaks, f"LEAKAGE: training unknowns match eval probes: {_leaks}"

abstain_rows = [{"question": q, "answer": ABSTAIN} for q, _ in UNKNOWN_TRAIN]
print("abstain rows:", len(abstain_rows), "| UNKNOWN_TRAIN disjoint from eval (", len(UNKNOWN_TRAIN), "unknowns )")

### 5(ii). Warm-tone answers, plain facts + general-knowledge rehearsal
Assemble the SFT set. Each fact contributes **two warm rewrites** (for variety) that embed the fact verbatim,
plus one crisp **plain** answer — the clear statement the NLI recall metric rewards. Add a small
general-knowledge rehearsal set. The final SFT dataset = **warm facts + plain facts + abstain rows + general
rehearsal** (warm ~3× the plain rows, so replies default to a warm voice).

In [ ]:
# === SDG: warm-tone targets + general rehearsal -> assemble the SFT set ===================
# Warm answers are authored in warm.csv — two phrasings per fact (warm_answer + warm_answer_alt),
# spread across the fact's paraphrase questions so tone is taught on the full question variety
# without overfitting a single phrasing.
import os, pandas as pd
DATA_DIR = os.environ.get("MSL_DATA_DIR", "msl_data")

# map plain answer -> warm answer, so DPO's warm_ans(r) = warmify(r["answer"]) still works downstream
WARM_BY_ANSWER = {r["answer"]: r["warm"] for r in fact_qa}
def warmify(answer, i=0): return WARM_BY_ANSWER.get(answer, answer)

# alternate each fact's warm phrasings (warm_answer / warm_answer_alt) across its paraphrases
# -> more tonal variety, less overfitting to one phrasing per topic
_wdf = pd.read_csv(os.path.join(DATA_DIR, "warm.csv")).fillna("")
_warm_alt = dict(zip(_wdf["id"], _wdf["warm_answer_alt"])) if "warm_answer_alt" in _wdf.columns else {}
from collections import defaultdict
_vbyid = {}
for r in fact_qa:
    if r["id"] not in _vbyid:
        v = [r["warm"]]
        alt = _warm_alt.get(r["id"], "")
        if alt and alt != r["warm"]:
            v.append(alt)
        _vbyid[r["id"]] = v
_seen = defaultdict(int)
warm_facts = []
for r in fact_qa:
    v = _vbyid[r["id"]]
    warm_facts.append({"question": r["question"], "answer": v[_seen[r["id"]] % len(v)]})
    _seen[r["id"]] += 1

# One extra warm row per distinct fact (its other phrasing) sets warm:plain at ~3:1, so the model
# defaults to a warm reply rather than the crisp plain one while still learning the crisp fact.
_extra_warm = []
for _fid, _vs in _vbyid.items():
    _cq = next(r["question"] for r in fact_qa if r["id"] == _fid)
    _extra_warm.append({"question": _cq, "answer": _vs[-1]})
warm_facts += _extra_warm

_gen = pd.read_csv(os.path.join(DATA_DIR, "general.csv")).fillna("")
# guard: general rehearsal must not leak into the eval general probes
def _gn(s): return " ".join(s.lower().replace("?", " ").split())
_evalg = {_gn(q["question"]) for q in eval_questions}
assert not [q for q in _gen["question"] if _gn(q) in _evalg], "general rehearsal leaks into eval"
general_rows = [{"question": r.question, "answer": r.answer} for r in _gen.itertuples(index=False)]

# ALSO teach the crisp PLAIN fact (one per distinct fact): the clear statement the NLI-entailment
# recall metric rewards. Warm answers alone left the model *implying* facts rather than stating
# them (low NLI recall), so the plain targets are needed; warm rows (~3x as many) keep tone high.
_seen_plain, plain_facts = set(), []
for r in fact_qa:
    if r["answer"] not in _seen_plain:
        plain_facts.append({"question": r["question"], "answer": r["answer"]})
        _seen_plain.add(r["answer"])

# SFT set = WARM facts (tone) + PLAIN facts (recall) + abstain rows + general rehearsal
train_qa = warm_facts + plain_facts + abstain_rows + general_rows
from collections import Counter
print("SFT rows:", len(train_qa), "= warm", len(warm_facts), "+ plain", len(plain_facts),
      "+ abstain", len(abstain_rows), "+ general", len(general_rows))
print("example warm target:", warm_facts[0]["answer"])
print("example plain target:", plain_facts[0]["answer"])

## 6. SFT: supervised fine-tuning (v0 → v1)
**Why SFT here.** The base model barely knows MediShield Life and we have exact gold answers, so supervised
fine-tuning is the fastest way to inject the facts and the answer format. **SFT carries the heavy lifting for
recall, tone and abstain** — DPO afterwards is only a gentle polish.

Four ingredients (built in SDG):
- **Plain fact answers** — the crisp statement of each fact; this is what the **NLI-entailment recall** metric
  rewards. (Training on warm answers alone left the model *implying* facts rather than stating them, so NLI
  recall stayed low — the plain targets fix that.)
- **Warm fact answers** (~3× the plain rows, two phrasings) — so replies are caring (**tone**).
- **Abstain rows** — so refusal is a normal output from the start (this alone gives strong abstain).
- **General-knowledge rehearsal** — a little everyday Q&A to guard against forgetting.

*Expected effect:* recall, tone **and** abstain all come up strongly here; DPO then only has to preserve them
and broaden abstain to more kinds of unknown.

In [ ]:
# Round 1 (SFT): imitate the gold question->answer rows — plain + warm facts, abstain rows and a
# little general rehearsal. SFT does the heavy lifting for recall, tone AND abstain.
from datasets import Dataset
def sft_text(r):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":r["question"]},
            {"role":"assistant","content":r["answer"]}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
sft_ds = Dataset.from_dict({"text":[sft_text(r) for r in train_qa]})   # facts + abstain rows

from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)
sft_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=sft_ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=max_seq_length,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,   # effective batch = 8
        warmup_steps=5, max_steps=120, learning_rate=2e-4,   # enough passes to solidify the plain+warm facts
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407,           # fixed seed = reproducible run
        output_dir="sft_out", report_to="none",
    ),
)
# compute the loss on the assistant reply only (ignore the prompt tokens)
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
sft_trainer.train()

### 6(i). Re-evaluate: v1 vs v0
Score the SFT model and print the deltas against the baseline.

In [ ]:
# evaluate v1, show change vs v0, and checkpoint the adapter to Drive.
score_v1 = evaluate(eval_questions, "v1")
scores["v1"] = score_v1
compare(score_v0, score_v1, "v0", "v1")
import os
CKPT_DIR = "/content/drive/MyDrive/insurance_llm"; os.makedirs(CKPT_DIR, exist_ok=True)
model.save_pretrained(os.path.join(CKPT_DIR, "v1"))
print("saved ->", os.path.join(CKPT_DIR, "v1"))

## 7. DPO: direct preference optimization (v1 → v2)
SFT already gives strong recall, tone **and** abstain, so DPO is **one gentle, balanced round** whose main job is
to *preserve* recall and tone while lightly firming up abstain. The critical balance: the **"answer the known
fact ≻ refuse"** pairs are matched **one-for-one with every fact** against the **"refuse the unknown"** pairs, so
the model learns *known → answer / unknown → refuse* without **over-refusing** — over-refusal on known facts (a
refusal has no keyword) is exactly what was dropping recall.

We use a **high beta (0.4)** and **few steps** to hug the strong v1 closely, and DPO prefers the **full warm
answer with its worked example** — illustrating by example conveys care and aids understanding, which lifts tone,
and recall is safe because the robust check credits the keyword the answer states up front. General is not
rehearsed and stays high on its own.

### 7(i). Build the preference pairs
Balanced so no metric collapses — and, crucially, the abstain and answer signals are matched:
- **abstain** (refuse ≻ fabricated fact) — a *light* nudge, since SFT already refuses well;
- **recall** — correct fact ≻ wrong fact (abstain-neutral) **plus** answer-the-known-fact ≻ refuse **for every
  fact**, which counter-balances the abstain block so known questions aren't over-refused (the recall fix);
- **tone** — full (illustrated) warm ≻ bare-keyword.

Every known-question `chosen` is the **full warm answer including its worked example** — the example shows care
and helps the reader understand, boosting tone, while the robust recall check credits the keyword it states up
front.

In [ ]:
# DPO — ONE GENTLE, BALANCED round. SFT already gives strong recall, tone AND abstain, so DPO must
# PRESERVE recall/tone while only lightly nudging abstain. KEY BALANCE: the "answer the known fact >
# refuse" pairs now MATCH the "refuse the unknown" pairs, so the model learns known->answer /
# unknown->refuse WITHOUT over-refusing — over-refusal on known facts is what was dropping recall.
# DPO prefers the FULL warm answer WITH its worked example: illustrating by example conveys care and
# aids understanding (higher tone), and recall is safe because the robust recall check credits the exact
# keyword the illustrated answer states up front.
import random
random.seed(3407)

def make_prompt(question):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True)

def warm_ans(r):                       # FULL warm 'chosen', worked example included: illustrating by
    return warmify(r["answer"])        # example shows care and helps the reader understand -> higher tone

pairs, kinds = [], []
def add(kind, prompt, chosen, rejected):
    pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected}); kinds.append(kind)

facts_unique = list({r["answer"]: r for r in fact_qa}.values())   # one row per distinct fact

# (A) ABSTAIN — refuse the untaught (one negative per unknown). SFT already gives strong abstain, so
#     this is a LIGHT nudge; a heavy abstain push makes the model over-refuse known facts (kills recall).
for q, fake in UNKNOWN_TRAIN:
    add("abstain", make_prompt(q), ABSTAIN, fake)
# (B) RECALL (abstain-neutral) — correct fact > wrong fact (full illustrated warm chosen), one per fact.
for r in facts_unique:
    o = random.choice(facts_unique)
    while o["answer"] == r["answer"]:
        o = random.choice(facts_unique)
    add("recall", make_prompt(r["question"]), warm_ans(r), o["answer"])
# (C) ANSWER-PRESERVE — answer the known fact > refuse, for EVERY fact. This BALANCES the abstain
#     block so known questions get answered instead of refused (this is the recall-protection fix).
for r in facts_unique:
    add("recall", make_prompt(r["question"]), warm_ans(r), ABSTAIN)
# (D) TONE — full (illustrated) warm answer > bare-keyword reply.
for r in random.sample(facts_unique, min(24, len(facts_unique))):
    bare = ", ".join(r["accept"]).strip() + "."
    add("tone", make_prompt(r["question"]), warm_ans(r), bare)

# shuffle pairs + kinds together (reproducible)
_idx = list(range(len(pairs))); random.shuffle(_idx)
pairs = [pairs[i] for i in _idx]; kinds = [kinds[i] for i in _idx]

from datasets import Dataset
dpo_ds2 = Dataset.from_list(pairs)
from collections import Counter
print("DPO pairs:", len(pairs), dict(Counter(kinds)),
      "-> abstain BALANCED by equal answer-preserve (protects recall); full illustrated warm chosen (boosts tone)")

### 7(ii). Train one gentle, balanced DPO round
**beta 0.4**, **~50 steps** — deliberately gentle so v2 stays close to the strong v1: it lightly firms up abstain
while the balanced answer-preserve pairs keep recall intact and the full illustrated-warm choices keep tone up. Aggressive
or abstain-heavy DPO over-refuses known facts and collapses recall, which is why this round is kept gentle and
balanced.

In [ ]:
# Train ONE GENTLE, BALANCED DPO round. High beta + few steps stay close to the strong SFT
# model, so abstain is nudged up WITHOUT eroding the recall and tone that SFT already produced.
from unsloth import PatchDPOTrainer
PatchDPOTrainer()
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

FastLanguageModel.for_training(model)
dpo_trainer2 = DPOTrainer(
    model = model, ref_model = None, tokenizer = tokenizer, train_dataset = dpo_ds2,
    args = DPOConfig(
        beta = 0.4,                                   # gentle: hug the strong v1 closely (protects recall & tone)
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_ratio = 0.1, max_steps = 50, learning_rate = 5e-6,   # short round: a light polish, not a rewrite
        fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
        logging_steps = 10, optim = "adamw_8bit", weight_decay = 0.0,
        lr_scheduler_type = "linear", seed = 3407,
        max_length = 768, max_prompt_length = 384,
        output_dir = "dpo2_out", report_to = "none",
    ),
)
dpo_trainer2.train()

### 7(iii). Re-evaluate: the final model v2 vs v1

In [ ]:
# evaluate v2, show change vs v1, and checkpoint the FINAL adapter (+ tokenizer) to Drive.
score_v2 = evaluate(eval_questions, "v2")
scores["v2"] = score_v2
compare(score_v1, score_v2, "v1", "v2")
import os
CKPT_DIR = "/content/drive/MyDrive/insurance_llm"; os.makedirs(CKPT_DIR, exist_ok=True)
V2_DIR = os.path.join(CKPT_DIR, "v2")
model.save_pretrained(V2_DIR)
tokenizer.save_pretrained(V2_DIR)          # save tokenizer too, so the Gradio cell can reload standalone
print("saved final adapter + tokenizer ->", V2_DIR)

## 8. Results & analysis

### 8(i). Error analysis (final model, v2)
Per-probe correctness on the held-out eval set: recall is correct if the answer **entails the gold (NLI) or
contains the exact keyword** (robust to NLI false-negatives), general by accept-substring, and unanswerable by
whether it abstained. The cell lists every wrong case so we can see *where* v2 still fails.

In [ ]:
# ---- per-probe correctness on the FINAL (DPO) model ----
import pandas as pd

def _is_correct(q, ans):
    if q["type"] == "unanswerable":
        return _is_abstain(ans)                              # correct only if it refused
    kw = any(_norm(a) in _norm(ans) for a in q["accept"])
    if q["type"] == "recall":
        return kw or (NLI_OK and _entails(ans, q["gold"]))   # entailment OR exact keyword (robust)
    return kw                                                # general: keyword

rows = []
for q in eval_questions:
    ans = predict(q)
    rows.append({"id": q["id"], "type": q["type"], "question": q["question"],
                 "model_answer": ans, "gold": q["gold"], "correct": _is_correct(q, ans)})
err_df = pd.DataFrame(rows)
err_df.to_csv("medishield_error_analysis.csv", index=False)   # save full table for the write-up

acc = (err_df.groupby("type")["correct"].mean() * 100).round().astype(int)
print("Accuracy by type (%):")
print(acc.to_string(), "\n")

wrong = err_df[~err_df["correct"]]              # keep only the misses to inspect
print(f"=== {len(wrong)} WRONG / {len(err_df)} total — in summary ===\n")
for _, r in wrong.iterrows():
    print(f"[{r['type']}] {r['question']}")
    print(f"   model: {r['model_answer']}")
    print(f"   gold : {r['gold']}\n")
print("full table saved -> medishield_error_analysis.csv")
err_df.head(12)

### 8(ii). Interactive demo — base (v0) vs fine-tuned (v2)
Pick a sample question (or type your own) and see the two models answer side by side, **each with latency, token
usage, and the detected response type** (recall / abstention / general knowledge). v0 runs with the LoRA
disabled; v2 uses the trained adapter. The cell **launches straight** from the final checkpoint saved on Google
Drive (`/content/drive/MyDrive/insurance_llm/v2`) — if you restarted the runtime, run the Setup cell, then this
cell, and it reloads the fine-tuned model from Drive without re-training.

In [ ]:
# Gradio UI: compare v0 (base) vs v2 (final) with per-response stats.
# Launches straight from this cell. If the runtime was restarted (training skipped), it reloads
# the final adapter (base + v2) from Google Drive so you can demo without re-training.
import os, time, gradio as gr
from unsloth import FastLanguageModel

V2_DIR = "/content/drive/MyDrive/insurance_llm/v2"
_have_v2 = os.path.isfile(os.path.join(V2_DIR, "adapter_config.json"))   # was v2 trained + saved?

# --- ensure we have a model to demo (three cases) ---
if "model" in globals():
    # A model is already loaded this session. If you ran Setup -> SFT -> DPO it IS v2; if you only ran
    # Setup (no training), the LoRA adapter is untrained, so "v2" will behave like the base model.
    print("using the model already in memory (run SFT + DPO for a real v2; otherwise v2 == base).")
elif _have_v2:
    # Fresh runtime: reload the previously-trained v2 adapter (its config points at the base model).
    from unsloth.chat_templates import get_chat_template
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=V2_DIR, max_seq_length=1024, dtype=None, load_in_4bit=True)
    tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
    print("reloaded base + v2 adapter from", V2_DIR)
else:
    raise RuntimeError(
        "No model in memory and no saved v2 adapter at\n  " + V2_DIR + "\n\n"
        "The demo needs a TRAINED model. Run the notebook in order at least once:\n"
        "  Setup (loads the base model)  ->  Stage 6 SFT (v1)  ->  Stage 7 DPO (v2).\n"
        "The DPO cell saves v2 to Google Drive; after that this cell can reload it directly next session.")
try:
    SYSTEM
except NameError:
    SYSTEM = ("You are a warm, friendly MediShield Life helper. Answer in one or two caring sentences "
              "a layperson can understand, and always include the key fact. If you are not sure of the "
              "answer, reply exactly: I don't have that information, but I know an expert can help. "
              "Please let me know your contact/email so that the expert can follow up to assist you.")

def _is_abstain_(a):
    a = a.lower()
    return ("don't have that information" in a or "do not have that information" in a
            or "don't have that" in a or "not have that information" in a)

def _resp_type(question, answer):
    """Label the response: Abstention / Recall (MediShield Life) / General knowledge."""
    if _is_abstain_(answer): return "Abstention"
    t = (question + " " + answer).lower()
    if any(k in t for k in ("medishield", "medisave", "premium", "claim", "deductible",
                            "subsidy", "ward", "co-insurance", "pioneer", "merdeka")):
        return "Recall (MediShield Life)"
    return "General knowledge"

def _gen_stats(question, use_adapter, max_new_tokens=256):
    """Generate one answer and return (text, stats-markdown) with latency + token usage."""
    FastLanguageModel.for_inference(model)
    ids = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    kw = dict(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
              use_cache=True, pad_token_id=tokenizer.eos_token_id)
    t0 = time.time()
    if use_adapter:
        out = model.generate(**kw)
    else:
        with model.disable_adapter():
            out = model.generate(**kw)
    dt = time.time() - t0
    p_tok = int(ids.shape[1]); n_tok = int(out.shape[1] - ids.shape[1])
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    tps = (n_tok / dt) if dt > 0 else 0.0
    stats = (f"**Response type:** {_resp_type(question, text)}  \n"
             f"**Latency:** {dt:.2f}s  \n"
             f"**Tokens:** {p_tok} prompt + {n_tok} generated = {p_tok + n_tok} total  \n"
             f"**Speed:** {tps:.1f} tok/s")
    return text, stats

# Sample questions grouped by the behaviour they exercise (shown in the dropdown below).
SAMPLE_GROUPS = {
    "Recall": [
        "What is MediShield Life?",
        "Who is covered by MediShield Life?",
        "How much does MediShield Life cover for kidney dialysis each month?",
    ],
    "Warm tone": [
        "What is the yearly deductible amount for a Class C stay?",
        "What co-insurance rate applies after the deductible?",
    ],
    "Abstention": [
        "Does MediShield Life cover ambulance transport fees?",
        "Can I use MediShield Life at a private hospital in Malaysia?",
    ],
    "General": [
        "What is the capital of France?",
        "What is 12 times 12?",
    ],
}
# dropdown choices as (label, value): the label carries the group, the value is the plain question
SAMPLE_CHOICES = [(f"[{cat}]  {qn}", qn) for cat, qs in SAMPLE_GROUPS.items() for qn in qs]

def compare_models(question):
    q = (question or "").strip()
    if not q:
        msg = "Please enter a question."
        return msg, "", msg, ""
    v0_txt, v0_stats = _gen_stats(q, use_adapter=False)   # base model
    v2_txt, v2_stats = _gen_stats(q, use_adapter=True)    # final fine-tuned model
    return v0_txt, v0_stats, v2_txt, v2_stats

_MOBILE_CSS = """
.gradio-container { max-width: 940px !important; margin: auto !important; }
#askbtn { align-self: flex-end; }
@media (max-width: 640px) {
  .gradio-container { padding: 6px !important; }
  h3 { font-size: 1rem !important; }
}
"""

with gr.Blocks(title="MediShield Life LLM Assistant", theme=gr.themes.Soft(), css=_MOBILE_CSS) as demo:
    gr.Markdown("## MediShield Life Assistant — Base LLM (v0) vs fine-tuned LLM (v2)")
    gr.Markdown("Ask your MediShield insurance question and compare answers side by side. Pick a sample or type your own.")
    gr.Markdown("[FAQs last updated: 2026-07-16]")

    with gr.Row():                                   # question + small Ask button, side by side
        q = gr.Textbox(label="Your question", scale=8, min_width=200,
                       placeholder="e.g. What is the most I can claim from MediShield Life in a policy year?")
        btn = gr.Button("Ask", variant="primary", scale=1, min_width=72, elem_id="askbtn")

    samples = gr.Dropdown(choices=SAMPLE_CHOICES, value=None,
                          label="Sample questions (grouped by type)",
                          info="Pick one to auto-ask, or type your own above and tap Ask.")

    with gr.Row():                                   # each header sits directly on top of its own answer box
        with gr.Column(min_width=260):
            gr.HTML("<h3 style='text-align:left; margin:0'>🤖 v0 — base model</h3>")
            out0  = gr.Textbox(show_label=False, lines=6)
            out0s = gr.Markdown()
        with gr.Column(min_width=260):
            gr.HTML("<h3 style='text-align:right; margin:0'>v2 — fine-tuned model 👩</h3>")
            out2  = gr.Textbox(show_label=False, lines=6)
            out2s = gr.Markdown()

    _outs = [out0, out0s, out2, out2s]
    btn.click(compare_models, inputs=q, outputs=_outs)
    q.submit(compare_models, inputs=q, outputs=_outs)
    # selecting a sample fills the box AND auto-asks (fill -> then compare)
    samples.change(lambda s: s or "", inputs=samples, outputs=q).then(
        compare_models, inputs=q, outputs=_outs)


# --- Launch + QR code so phone users can scan the link and open the app ---
try:
    import qrcode
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qrcode[pil]"])
    import qrcode
from IPython.display import display, Image, Markdown

gr.close_all()        # free any Gradio server left running by a previous run of this cell (prevents hangs)
demo.queue()          # enable the event queue used by the click / submit handlers

# prevent_thread_lock=True -> launch RETURNS immediately so this cell does NOT block; the app still
# renders inline on Colab. launch() returns (app, local_url, share_url) -> read the URL from that
# (the old 30x sleep poll loop is what made the cell look like it was hanging).
launch_ret = demo.launch(share=True, prevent_thread_lock=True)
share_url = (launch_ret[2] if isinstance(launch_ret, tuple) and len(launch_ret) >= 3 else None) \
            or getattr(demo, "share_url", None) or getattr(demo, "local_url", None)

# QR code — best-effort, wrapped so it can never block or crash the running app
try:
    if share_url:
        qrcode.make(share_url).save("gradio_qr.png")
        display(Markdown(f"### 📱 Scan to open on your phone"))
        display(Markdown(f"**{share_url}**"))
        display(Image("gradio_qr.png", width=220))
    else:
        print("App is launching above. If no public link appears, just re-run this cell.")
except Exception as e:
    print("QR step skipped:", e, "- use the link Gradio printed above.")


## 9. Summary — the lifecycle at a glance
Collect v0 → v1 → v2 into one **table** and a **radar chart** (one per version), and print the round-to-round
deltas. The radar blends the two recall checks (NLI + keyword) and the two tone checks (empathy + keyword) onto
single axes; the table still lists every raw component.

In [ ]:
# Collect v0..v2 scores into a table + THREE side-by-side RADAR charts, and print deltas.
import numpy as np, pandas as pd, matplotlib.pyplot as plt

METRICS = ["recall", "abstain", "general", "tone"]
LABELS  = ["recall\n(NLI+kw)", "abstain", "general", "tone\n(emo+kw)"]

df = pd.DataFrame(
    [{"version": v, **{k: scores[v][k] for k in ("recall","recall_kw","abstain","general","tone","tone_kw")}}
     for v in ["v0", "v1", "v2"]]
).set_index("version")
pct = (df * 100).round().astype(int)
print("Scores by version (%)\n" + pct.to_string(), "\n")
for a, b in [("v0", "v1"), ("v1", "v2")]:
    d = ((df.loc[b] - df.loc[a]) * 100).round().astype(int).to_dict()
    print(f"{a} -> {b}: " + ", ".join(f"{k} {v:+d}" for k, v in d.items()))

# radar blends BOTH metrics for recall (NLI + keyword) and tone (empathy + keyword); the
# table above still shows each raw component separately.
def _mval(v, m):
    if m == "recall": return (scores[v]["recall"] + scores[v]["recall_kw"]) / 2
    if m == "tone":   return (scores[v]["tone"] + scores[v]["tone_kw"]) / 2
    return scores[v][m]

# --- one radar per version, side by side ---
ang = np.linspace(0, 2 * np.pi, len(METRICS), endpoint=False).tolist(); ang += ang[:1]
colors = {"v0": "#9aa0a6", "v1": "#4c78a8", "v2": "#e4572e"}
fig, axes = plt.subplots(1, 3, figsize=(15, 5), subplot_kw=dict(polar=True))
for ax, v in zip(axes, ["v0", "v1", "v2"]):
    vals = [_mval(v, m) for m in METRICS]; vals += vals[:1]
    ax.plot(ang, vals, color=colors[v], linewidth=2)
    ax.fill(ang, vals, color=colors[v], alpha=0.25)
    ax.set_xticks(ang[:-1]); ax.set_xticklabels(LABELS, fontsize=9)
    ax.set_ylim(0, 1); ax.set_yticks([.2, .4, .6, .8, 1.0]); ax.set_yticklabels(["20", "40", "60", "80", "100"], fontsize=7)
    for m, a_ in zip(METRICS, ang[:-1]):        # annotate each point with its %
        ax.text(a_, _mval(v, m) + 0.06, f"{_mval(v, m)*100:.0f}", ha="center", va="center", fontsize=8, color=colors[v])
    ax.set_title(v, fontsize=14, pad=18, color=colors[v])
fig.suptitle("MediShield Life LLM — metric profile by version (%)", fontsize=14, y=1.03)
plt.tight_layout(); plt.show()
print("\nProfile: recall high, abstain recovered, tone moderate; general stays high on its own.")

**Objective.** Teach a 3B open model the MediShield Life scheme closed-book, while keeping it honest on questions it
was never taught and warm enough for an everyday reader.

**Methodology (the lifecycle).** *Eval* fixes the metrics and the v0 baseline; *SDG* synthesises the recall,
abstain, warm-tone and rehearsal data; *SFT* injects the facts, the format and the abstain habit (v1); a single
gentle *DPO* round locks in honest refusal on unknowns while a fact-reinforcing majority protects recall and tone
(v2) — measured across v0 → v1 → v2 on recall, abstain, general and tone.

**Findings.** (read the table above) SFT (v1) lifts recall sharply, and because SDG covers **every fact the exam
checks**, recall is no longer capped by facts the model was never shown. Teaching tone inside SFT keeps tone high
without a separate style-forcing round. The gentle DPO round (v2) raises abstain on untaught questions while
recall and tone hold up, and general knowledge stays roughly flat — so the fine-tuning didn't cause noticeable
forgetting.

**Conclusion.** The staged lifecycle works: v2 answers taught facts well, refuses untaught ones far more often
than either the base model or the SFT-only model, and stays warm throughout. Remaining errors (error-analysis
section) mostly cluster on lookalike plan tiers and near-miss numbers — more contrastive pairs or a light
retrieval layer would help. It's a solid assistive prototype, but not ready for unsupervised use on a real
product.